<a href="https://colab.research.google.com/github/Hithavan31/BIS_PROGRAMS/blob/main/1BM24CS118__Wireless_sensor_and_network_optimization__Particle_Swarm_Optimization_Problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# --- 1. NETWORK & ENERGY PARAMETERS ---
NUM_NODES = 20            # Total sensor nodes in the field
FIELD_SIZE = 100          # 100m x 100m field
NUM_CH_TARGET = 4         # Target number of Cluster Heads

# Base Station (Server Node) coordinates at center-top
BS_X, BS_Y = 50.0, 100.0

# Radio energy model parameters (Standard LEACH protocol metrics)
E_INIT = 2.0              # Initial energy of each node in Joules
E_TX = 50e-9              # Energy to transmit 1 bit (50 nJ/bit)
E_RX = 50e-9              # Energy to receive 1 bit (50 nJ/bit)
E_DA = 5e-9               # Data aggregation energy (5 nJ/bit/signal)
AMP_FREE_SPACE = 10e-12   # Free space amplifier coefficient (10 pJ/bit/m^2)
AMP_MULTIPATH = 0.0013e-12# Multipath amplifier coefficient (0.0013 pJ/bit/m^4)
D0 = np.sqrt(AMP_FREE_SPACE / AMP_MULTIPATH)  # Threshold distance (approx 87m)
PACKET_BITS = 4000        # Packet size (4000 bits)

# Generate static random node positions and uniform residual energies for evaluation
np.random.seed(42)
NODE_POSITIONS = np.random.uniform(0, FIELD_SIZE, (NUM_NODES, 2))
NODE_ENERGY = np.random.uniform(0.5, E_INIT, NUM_NODES)  # Current residual energy state

# --- 2. MULTI-OBJECTIVE FITNESS FUNCTION ---
def calculate_network_energy_cost(position):
    """
    Evaluates particle configuration.
    Position Vector Length = NUM_NODES.
    Values 0.0 to 1.0: Nodes with the top 'NUM_CH_TARGET' highest values become CHs.
    """
    # Step A: Identify Cluster Heads (CH) based on highest continuous weights
    ch_indices = np.argsort(position)[-NUM_CH_TARGET:]
    non_ch_indices = np.array([i for i in range(NUM_NODES) if i not in ch_indices])

    total_energy_dissipated = 0.0

    # Check if any elected CH is completely dead
    for ch in ch_indices:
        if NODE_ENERGY[ch] <= 0:
            return 1e9  # Hard penalty: cannot elect dead nodes as CHs

    # Step B: Non-CH Nodes transmit data to their nearest elected CH
    for node in non_ch_indices:
        # Find closest CH
        distances = np.linalg.norm(NODE_POSITIONS[ch_indices] - NODE_POSITIONS[node], axis=1)
        closest_ch_idx = ch_indices[np.argmin(distances)]
        min_dist = np.min(distances)

        # Energy spent by regular node transmitting to CH
        if min_dist < D0:
            e_tx = PACKET_BITS * (E_TX + AMP_FREE_SPACE * (min_dist ** 2))
        else:
            e_tx = PACKET_BITS * (E_TX + AMP_MULTIPATH * (min_dist ** 4))

        total_energy_dissipated += e_tx

        # Energy spent by the target CH receiving this node's packet
        total_energy_dissipated += PACKET_BITS * E_RX

    # Step C: CH Nodes aggregate data and route to the Base Station Server
    for ch in ch_indices:
        # Data aggregation energy overhead
        total_energy_dissipated += PACKET_BITS * E_DA * (NUM_NODES / NUM_CH_TARGET)

        # Route distance directly to the server node (Base Station)
        dist_to_bs = np.linalg.norm(NODE_POSITIONS[ch] - np.array([BS_X, BS_Y]))

        if dist_to_bs < D0:
            e_bs = PACKET_BITS * (E_TX + AMP_FREE_SPACE * (dist_to_bs ** 2))
        else:
            e_bs = PACKET_BITS * (E_TX + AMP_MULTIPATH * (dist_to_bs ** 4))

        total_energy_dissipated += e_bs

    # Step D: Multi-objective formulation
    # Objective 1: Minimize current dissipated energy round
    # Objective 2: Maximize structural lifetime balance (penalize choosing low energy nodes as CH)
    ch_residual_energy = NODE_ENERGY[ch_indices]
    energy_balance_penalty = np.sum(E_INIT - ch_residual_energy) * 0.1

    fitness_score = total_energy_dissipated + energy_balance_penalty
    return fitness_score, ch_indices

# --- 3. PSO ENGINE CONFIGURATION ---
POP_SIZE = 30
MAX_ITER = 60
W = 0.6
C1 = 2.0
C2 = 1.8

# Boundaries map to probabilities / rank arrays
X = np.random.uniform(0.0, 1.0, (POP_SIZE, NUM_NODES))
V = np.random.uniform(-0.2, 0.2, (POP_SIZE, NUM_NODES))

P_best = np.copy(X)
P_best_fitness = np.full(POP_SIZE, fill_value=1e9)

for i in range(POP_SIZE):
    cost, _ = calculate_network_energy_cost(X[i])
    P_best_fitness[i] = cost

g_best_index = np.argmin(P_best_fitness)
G_best = np.copy(P_best[g_best_index])
G_best_fitness = P_best_fitness[g_best_index]

# --- 4. OPTIMIZATION LOOP ---
for iteration in range(MAX_ITER):
    for i in range(POP_SIZE):
        r1, r2 = np.random.rand(NUM_NODES), np.random.rand(NUM_NODES)

        # Standard velocity math using network metrics
        V[i] = W * V[i] + C1 * r1 * (P_best[i] - X[i]) + C2 * r2 * (G_best - X[i])
        V[i] = np.clip(V[i], -0.2, 0.2)

        X[i] = X[i] + V[i]
        X[i] = np.clip(X[i], 0.0, 1.0)

        current_fitness, _ = calculate_network_energy_cost(X[i])

        if current_fitness < P_best_fitness[i]:
            P_best_fitness[i] = current_fitness
            P_best[i] = np.copy(X[i])

            if current_fitness < G_best_fitness:
                G_best_fitness = current_fitness
                G_best = np.copy(X[i])

# --- 5. REPORT RUN PERFORMANCE ---
final_energy_score, optimal_ch_nodes = calculate_network_energy_cost(G_best)

print("============ WSN SWARM OPTIMIZATION COMPLETE ============")
print(f"Global Fitness Score Minimized to : {final_energy_score:.6f}")
print(f"Selected Server Cluster Heads (Node IDs): {sorted(optimal_ch_nodes.tolist())}")
print("\nIndividual Residual Energy Check of Selected CH Nodes:")
for ch in sorted(optimal_ch_nodes):
    print(f"  * Node ID {ch:02d} | Remaining Node Energy: {NODE_ENERGY[ch]:.4f} Joules")


============ WSN SWARM OPTIMIZATION COMPLETE ============
Global Fitness Score Minimized to : 0.047838
Selected Server Cluster Heads (Node IDs): [3, 10, 12, 15]

Individual Residual Energy Check of Selected CH Nodes:
  * Node ID 03 | Remaining Node Energy: 1.8640 Joules
  * Node ID 10 | Remaining Node Energy: 1.9544 Joules
  * Node ID 12 | Remaining Node Energy: 1.9092 Joules
  * Node ID 15 | Remaining Node Energy: 1.8828 Joules
